<a href="https://colab.research.google.com/github/SyChen94/colab/blob/main/defuse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/chunlinli/defuse.git

Cloning into 'defuse'...
remote: Enumerating objects: 447, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 447 (delta 22), reused 20 (delta 9), pack-reused 407 (from 1)
Receiving objects: 100% (447/447), 68.53 MiB | 19.04 MiB/s, done.
Resolving deltas: 100% (111/111), done.


In [26]:
!cd defuse

/bin/bash: line 1: cd: defuse: No such file or directory


In [27]:
%cd /content/defuse
!ls
!find . -maxdepth 3 -name "setup.py" -o -name "pyproject.toml"


/content/defuse
defuse		     example_small.ipynb  simulation_hub_large.csv
defuse.egg-info      LICENSE		  simulation_hub_small.csv
defuse.png	     README.md		  simulation_random_large.csv
environment.yml      setup.py		  simulation_random_small.csv
example_large.ipynb  simulation
./setup.py


In [28]:
%cd /content/defuse

!python -m pip install -U pip setuptools wheel
!python -m pip install -e .


/content/defuse
Obtaining file:///content/defuse
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for defuse (pyproject.toml) ... done
  Created wheel for defuse: filename=defuse-0.1-0.editable-py3-none-any.whl size=6861 sha256=27ef8385c63af144f64b3eb352bd20a026e051d04a59e98ecafa17512e753399
  Stored in directory: /tmp/pip-ephem-wheel-cache-tltz4wt4/wheels/d4/27/d0/ba57b9e57ca3a8f69f665139ee08cc3a152ba017514d7e222c
Successfully built defuse
  Attempting uninstall: defuse
    Found existing installation: defuse 0.1
    Uninstalling defuse-0.1:
      Successfully uninstalled defuse-0.1


In [ ]:
! pip install igraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 31.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [igraph]


In [31]:
import defuse, os
print(defuse.__file__)


/content/defuse/defuse/__init__.py


In [33]:
%cd /content/defuse
!python -m pip install -e .


/content/defuse
Obtaining file:///content/defuse
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for defuse (pyproject.toml) ... done
  Created wheel for defuse: filename=defuse-0.1-0.editable-py3-none-any.whl size=6861 sha256=230080bef217b21b1da563f9a45db6acf88e48655714b4325759e418a7eb93ff
  Stored in directory: /tmp/pip-ephem-wheel-cache-b93t4cxa/wheels/d4/27/d0/ba57b9e57ca3a8f69f665139ee08cc3a152ba017514d7e222c
Successfully built defuse
  Attempting uninstall: defuse
    Found existing installation: defuse 0.1
    Uninstalling defuse-0.1:
      Successfully uninstalled defuse-0.1


In [34]:
import sys
if "defuse" in globals():
    del globals()["defuse"]
for m in list(sys.modules.keys()):
    if m == "defuse" or m.startswith("defuse."):
        del sys.modules[m]


In [35]:
from defuse.defuse import Defuse
import defuse
print("defuse file:", defuse.__file__)
print("✅ Defuse imported OK")


defuse file: /content/defuse/defuse/__init__.py
✅ Defuse imported OK


In [63]:
# ==============================
# Utilities: stability
# ==============================
import numpy as np
from numpy.linalg import eigvals

def spectral_radius(B):
    return np.max(np.abs(eigvals(B)))

def make_stable_beta(B, radius_max=0.8):
    r = spectral_radius(B)
    if np.isfinite(r) and r > 0 and r >= radius_max:
        B = (radius_max / r) * B
    return B

# ==============================
# Step 1: DAG generation (true B + true topological order)
# ==============================
def generate_dag_structure(p, density=0.2,
                           a=3.3, b=3.8,
                           radius_max=0.8,
                           seed=None):
    if seed is not None:
        np.random.seed(seed)

    pi_order = np.random.permutation(p)
    B = np.zeros((p, p))

    for ii in range(p - 1):
        for jj in range(ii + 1, p):
            if np.random.rand() < density:
                i = pi_order[ii]
                j = pi_order[jj]
                B[i, j] = np.random.uniform(a, b) * np.random.choice([-1, 1])

    B = make_stable_beta(B, radius_max)
    return B, pi_order

def generate_hidden_confounders(n, s):
    if s <= 0:
        return np.zeros((n, 0))
    return np.random.randn(n, s)

def generate_phi(s, p, phi_strength):
    if s <= 0 or phi_strength == 0:
        return np.zeros((0, p))
    return phi_strength * np.random.randn(s, p)
def simulate_Y_linear_sem(B, n,
                          s=0,
                          phi_strength=0.0,
                          noise_var=0.5,
                          standardize=True,
                          seed=None):
    if seed is not None:
        np.random.seed(seed)

    p = B.shape[0]

    # latent confounders
    H = generate_hidden_confounders(n, s)
    Phi = generate_phi(s, p, phi_strength)

    # i.i.d. noise
    E = np.sqrt(noise_var) * np.random.randn(n, p)

    # (I - B)^{-1}
    IminusB = np.eye(p) - B
    if np.linalg.cond(IminusB) > 1e10:
        raise ValueError("(I - B) nearly singular")

    IB_inv = np.linalg.inv(IminusB)

    Y = (H @ Phi + E) @ IB_inv

    if standardize:
        Y = (Y - Y.mean(axis=0)) / Y.std(axis=0)

    return Y


In [64]:
Y_no = simulate_Y_linear_sem(B_true, n, s=0, phi_strength=0.0, seed=2)
Y_cf = simulate_Y_linear_sem(B_true, n, s=5, phi_strength=0.5, seed=2)

# ---- run defuse ----
m = Defuse()
A_est_no, _ = m.fit(Y_no, verbose=False)

m = Defuse()
A_est_cf, _ = m.fit(Y_cf, verbose=False)

print("No confounding:", count_accuracy((B_true!=0).astype(int), A_est_no))
print("With confounding:", count_accuracy((B_true!=0).astype(int), A_est_cf))

No confounding: {'fdr': 0.0, 'tpr': 0.0, 'fpr': 0.0, 'shd': 42, 'nnz': 0}
With confounding: {'fdr': 0.0, 'tpr': 0.0, 'fpr': 0.0, 'shd': 42, 'nnz': 0}


In [68]:
import numpy as np
from defuse.utils import count_accuracy

def er_dag(p, rho, rng):
    # generate upper-triangular adjacency under a random topo order
    order = rng.permutation(p)
    A = np.zeros((p, p), dtype=int)
    for ii in range(p-1):
        for jj in range(ii+1, p):
            if rng.random() < rho:
                A[order[ii], order[jj]] = 1
    # return A and the topo order (as indices 0..p-1 in causal order)
    # We need a topo list to simulate: use inverse map of order positions
    topo = order.tolist()
    return A, topo

def make_beta_from_A(A, a=0.3, b=0.8, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    p = A.shape[0]
    B = np.zeros((p,p))
    idx = np.where(A==1)
    signs = rng.choice([-1.0, 1.0], size=len(idx[0]))
    mags = rng.uniform(a, b, size=len(idx[0]))
    B[idx] = signs * mags
    return B

def correlated_noise(n, p, rng, strength=1.0):
    # mimic defuse default: banded-ish correlation
    cov = np.zeros((p,p))
    off = np.zeros(p-1)
    off[::2] = 1.0 * strength
    cov[np.arange(p-1), np.arange(1,p)] = off
    cov[np.arange(1,p), np.arange(p-1)] = off
    np.fill_diagonal(cov, 2.0)
    Z = rng.multivariate_normal(np.zeros(p), cov, size=n)
    return Z

def simulate_Y_like_R(A, B, topo, n=500, sem="poly-trig",
                      noise_type="correlated", noise_var=1.0, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    p = A.shape[0]

    # noise Z: correlated or iid
    if noise_type == "correlated":
        Z = correlated_noise(n, p, rng, strength=1.0)
    else:
        Z = rng.normal(0, np.sqrt(noise_var), size=(n,p))

    Y = np.zeros((n,p))

    def f(x):
        # defuse-like nonlinear
        # square/cos per parent
        return x

    for j in topo:  # topo order
        pa = np.where(A[:, j]==1)[0]
        if len(pa)==0:
            Y[:, j] = Z[:, j]
        else:
            if sem == "linear":
                Y[:, j] = Y[:, pa] @ B[pa, j] + Z[:, j]
            elif sem == "poly-trig":
                # per-parent random square/cos with stronger coefficients
                out = np.zeros((n,1))
                for k in pa:
                    if rng.random() < 0.5:
                        out += (2.8*np.sign(B[k,j])) * (Y[:, [k]]**2)
                    else:
                        out += (2.8*np.sign(B[k,j])) * np.cos(Y[:, [k]])
                Y[:, j] = out.ravel() + Z[:, j]
            else:
                raise ValueError("unknown sem")
    return Y

# ---- run one experiment ----
rng = np.random.default_rng(1)
p, n, rho = 20, 500, 0.2

A_true, topo = er_dag(p, rho, rng)
B_true = make_beta_from_A(A_true, a=0.3, b=0.8, rng=rng)

Y = simulate_Y_like_R(A_true, B_true, topo, n=n, sem="linear",
                      noise_type="uncorrelated", rng=rng)

A_est, _ = Defuse().fit(Y, verbose=False)

print("true nnz:", int(A_true.sum()))
print("est nnz :", int((A_est!=0).sum()))
print(count_accuracy(A_true, A_est))


true nnz: 30
est nnz : 0
{'fdr': 0.0, 'tpr': 0.0, 'fpr': 0.0, 'shd': 30, 'nnz': 0}


In [44]:
import numpy as np

from defuse.utils import set_random_seed, simulate_dag, simulate_data, count_accuracy

set_random_seed(1110)
d, n = 20, 500
A = simulate_dag(d, 'random')
X, cf, fn = simulate_data(A, n, 'poly-trig')

A_est, _ = Defuse().fit(X, verbose=False)
print(count_accuracy(A, A_est))
print("nnz(A_est) =", (A_est!=0).sum())

{'fdr': 0.0, 'tpr': 1.0, 'fpr': 0.0, 'shd': 0, 'nnz': 3}
nnz(A_est) = 3
